# 문서 압축과 문맥 압축

문서 압축은 문서 전체를 미리 요약하거나 정제해 더 짧은 표현으로 인덱싱하는 방식이다. 문맥 압축은 사용자의 질의가 들어온 뒤 검색된 문서에서 그 질문과 관련된 구절만 추출하거나 요약하는 방식이다. 전자는 질의와 무관한 오프라인 변환이고, 후자는 질의별 온라인 변환이라는 경계가 있다.

이 단원은 핵심 검색 경로를 익힌 뒤 필요에 따라 선택하는 독립 심화 분기이다. 실행에는 Dense 검색·임베딩·Pinecone 인덱싱의 기본 개념과 OpenAI·Pinecone 설정이 필요하며 패키지와 CSV는 이 노트북에서 준비한다. HyDE나 Cohere Rerank의 출력을 사용하지 않으므로 세 기법을 순서대로 실행하는 직렬 파이프라인으로 연결되지 않는다.

이 실습의 코드는 `원문 → LLM 요약 → 압축 문서 인덱스 → 질의 검색`을 구현하므로 정확히는 인덱싱 시점의 문서 압축이다. 출력 벡터 차원은 줄지 않지만 임베딩에 반영되는 텍스트와 정보 밀도가 달라진다. 장점은 중복 표현과 입력 길이를 줄일 수 있다는 점이고, 한계는 요약 과정에서 고유명사·수치·조건이 사라질 수 있고 전체 문서에 LLM 비용이 든다는 점이다. 원문 인덱스와 압축 인덱스를 Recall@5, MRR와 MAP로 비교해 손실 여부를 확인한다.


## 문서 압축 실행 패키지 준비

`%pip`은 현재 Jupyter 커널에 필요한 패키지를 설치한다. OpenAI 요약 체인과 원문·압축 Pinecone 인덱스를 준비한다. 설치 뒤 커널이 이전 모듈을 계속 사용하면 한 번 재시작한다.

### 코드 해석 순서

1. 문서 압축 실습에 필요한 공식 패키지를 현재 커널에 설치한다.

### 결과 해석

- 패키지별 설치 로그가 나타나며 의존성 충돌이 없으면 셀이 종료된다.
- 설치가 끝나면 다음 셀에서 문서 압축의 모델·검색기 객체를 직접 import할 수 있다.


In [1]:
# 설치 목록은 문서 압축 활성 코드에 필요한 SDK와 분석 도구를 포함한다.
# `-U`는 이미 설치된 패키지를 호환되는 최신 배포본으로 갱신한다.
# %pip install -U pandas numpy langchain langchain-openai langchain-pinecone pinecone python-dotenv gdown tqdm


  Using cached langchain_openai-1.5.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached pinecone-9.1.0-cp310-abi3-win_amd64.whl.metadata (6.3 kB)
   ---------------------------------------- 0.0/12.5 MB ? eta -:--:--
    --------------------------------------- 0.3/12.5 MB ? eta -:--:--
   -------------- ------------------------- 4.5/12.5 MB 16.1 MB/s eta 0:00:01
   ------------------------- -------------- 7.9/12.5 MB 15.9 MB/s eta 0:00:01
   ----------------------------------- ---- 11.0/12.5 MB 15.4 MB/s eta 0:00:01
   ---------------------------------------  12.3/12.5 MB 15.9 MB/s eta 0:00:01
   ---------------------------------------  12.3/12.5 MB 15.9 MB/s eta 0:00:01
   ---------------------------------------- 12.5/12.5 MB 9.3 MB/s  0:00:01
Using cached langchain_openai-1.5.0-py3-none-any.whl (123 kB)

  Attempting uninstall: numpy

    Found existing installation: numpy 2.5.1

   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [n

ERROR: Could not install packages due to an OSError: [WinError 2] 지정된 파일을 찾을 수 없습니다



## 문서 압축 환경 변수와 모델 이름 준비

`load_dotenv()`는 `.env`에 저장한 OpenAI·Pinecone 설정을 현재 Python 환경으로 불러온다. API key는 출력하지 않으며 각 SDK가 환경 변수에서 직접 사용한다.

이 노트북은 다음 설정으로 문서 요약과 두 Pinecone index의 검색 결과를 비교한다.

- `OPENAI_LLM_MODEL`: 원문을 요약할 Chat Model이다.
- `OPENAI_EMBEDDING_MODEL`: 원문·요약문·질의를 같은 벡터 공간으로 바꾼다.
- `PINECONE_INDEX_NAME`: 원문이 저장된 기준 index이다.
- `PINECONE_COMP_INDEX_NAME`: 요약 문서를 저장할 비교 index이다.
- `PINECONE_INDEX_DIMENSION`, `PINECONE_INDEX_METRIC`: 두 index가 사용할 벡터 길이와 유사도 기준이다.

### 코드 해석 순서

1. `.env`를 불러오고 문서 압축과 Pinecone 비교에 사용할 설정을 준비한다.

### 결과 해석

- 환경 변수와 실습용 모델·index 설정이 준비되며 화면에 별도 출력은 나타나지 않는다.
- 두 index의 설정을 같게 두고 저장 본문만 바꾸면 문서 압축의 영향만 비교할 수 있다.


In [2]:
import os
from dotenv import load_dotenv

# load_dotenv()는 `.env`의 값을 환경 변수에 추가하며 API key를 화면에 출력하지 않는다.
load_dotenv()

# 생성 모델은 원문을 요약하고 임베딩 모델은 원문·요약문·질의를 벡터로 바꾼다.
OPENAI_LLM_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")
OPENAI_EMBEDDING_MODEL = os.getenv(
    "OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"
)

# 원문 index와 압축 index의 이름을 분리해 저장할 텍스트 표현만 다르게 비교한다.
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "adv-rag")
PINECONE_COMP_INDEX_NAME = os.getenv(
    "PINECONE_COMP_INDEX_NAME", "adv-rag-compressed"
)

# 새 압축 index는 원문 index와 같은 벡터 차원과 거리 계산 기준을 사용한다.
PINECONE_INDEX_REGION = os.getenv("PINECONE_INDEX_REGION", "us-east-1")
PINECONE_INDEX_CLOUD = os.getenv("PINECONE_INDEX_CLOUD", "aws")
PINECONE_INDEX_METRIC = os.getenv("PINECONE_INDEX_METRIC", "cosine")
PINECONE_INDEX_DIMENSION = int(os.getenv("PINECONE_INDEX_DIMENSION", "1536"))


## 원문·질의 CSV 다운로드

압축할 문서와 검색 평가용 질의·정답 파일을 저장한다. 원문 index와 압축 index는 같은 `doc_id`와 질문을 사용해야 텍스트 변환의 영향만 비교할 수 있다. 이 셀의 파일은 다음 DataFrame과 요약 체인의 공통 입력이 된다.

### 코드 해석 순서

1. 문서 압축과 검색 평가에 사용할 두 CSV를 내려받는다.

### 결과 해석

- 실행하면 문서와 질의 파일의 다운로드 완료 진행률이 나타난다.
- 같은 문서 ID를 유지하면 요약 전후 검색 결과를 문서 단위로 직접 대조할 수 있다.


In [3]:
# `documents.csv`의 본문은 요약 체인의 입력이며 원문 기준선도 유지한다.
# `queries.csv`는 두 인덱스를 같은 질문과 qrels로 평가하게 한다.
# documents.csv
!gdown 1pspaw_q4_QCp2K4M-thDlUtra-_wQI7k
# queries.csv
!gdown 1dsn0pwkfzOUxiQ4MKDIvMM-CkxbYiSce


Downloading...
From: https://drive.google.com/uc?id=1pspaw_q4_QCp2K4M-thDlUtra-_wQI7k
To: C:\SKN_AI\09_llm\07_advanced_rag\01_retrieval_optimizationi\documents.csv

  0%|          | 0.00/14.9k [00:00<?, ?B/s]
100%|██████████| 14.9k/14.9k [00:00<00:00, 15.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1dsn0pwkfzOUxiQ4MKDIvMM-CkxbYiSce
To: C:\SKN_AI\09_llm\07_advanced_rag\01_retrieval_optimizationi\queries.csv

  0%|          | 0.00/2.19k [00:00<?, ?B/s]
100%|██████████| 2.19k/2.19k [00:00<00:00, 3.75MB/s]


## 원문과 질의 테이블 로드

두 CSV를 DataFrame으로 읽고 첫 문서 본문을 확인한다. `documents_df`의 한 행은 원문 하나이고, `queries_df`의 한 행은 나중에 두 index에 공통으로 보낼 질문 하나이다. 첫 본문 문자열은 다음 셀에서 요약 전 길이와 결과를 비교하는 대표 입력이 된다.

### 코드 해석 순서

1. 문서와 질의 DataFrame을 만들고 대표 원문을 확인한다.

### 결과 해석

- 실행하면 제주도 관광 정보를 담은 D1 원문 전체가 문자열로 표시된다.
- 대표 원문의 고유명사·수치·조건을 기억해야 요약에서 정보가 빠졌는지 판단할 수 있다.


In [4]:
# `documents_df.loc[0, 'content']`는 D1의 긴 원문 문자열을 반환한다.
# `queries_df`는 뒤의 원문·압축 인덱스 검색 반복문에서 사용된다.
import pandas as pd

documents_df = pd.read_csv('documents.csv')
queries_df = pd.read_csv('queries.csv')

documents_df.loc[0, 'content']


'제주도는 대한민국의 대표 관광지로서, 한라산 등반, 성산 일출봉 관광, 해변 활동(협재해수욕장·함덕해수욕장) 등이 인기입니다. 현지 음식으로는 흑돼지, 고기국수, 전복죽 등이 있으며, 카페 거리(서귀포시 대정읍 카페 거리)도 유명합니다. 교통은 렌터카나 시외버스를 주로 이용하며, 사전 예약 시 우도 투어나 올레길 트레킹도 즐길 수 있습니다.'

## 대표 문서 요약과 길이 비교

`PromptTemplate → ChatOpenAI → StrOutputParser` 체인은 원문 문자열을 짧은 요약 문자열로 바꾼다.

- `PromptTemplate`: 검색에 필요한 고유명사·수치·조건을 보존하도록 요약 기준을 전달한다.
- `ChatOpenAI`: 완성된 프롬프트를 받아 요약을 생성한다.
- `StrOutputParser`: 모델의 메시지에서 요약 문자열만 꺼낸다.

대표 문서의 원문과 요약을 함께 출력한다. 문자 수가 줄었는지뿐 아니라 검색에 필요한 정보가 남았는지도 확인한다.

### 코드 해석 순서

1. text에 들어온 원문에서 검색에 중요한 정보를 남기도록 요약 지시문을 만든다.
2. ChatOpenAI는 프롬프트를 요약 메시지로 바꾸고 Parser는 이를 문자열로 변환한다.
3. 첫 문서 원문을 체인에 전달하면 요약 문자열이 반환된다.

### 결과 해석

- OpenAI 호출 뒤 원문과 요약문의 문자 수 및 전체 문자열이 차례대로 출력된다.
- 요약이 짧아져도 고유명사·수치·조건이 사라지면 검색 문서로 사용하기 어렵다.


In [5]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 1. text에 들어온 원문에서 검색에 중요한 정보를 남기도록 요약 지시문을 만든다.
summary_prompt = PromptTemplate.from_template('''
다음 문서의 검색 핵심어, 고유명사, 수치와 조건을 보존하며 간결하게 요약한다.

문서:
{text}

요약:
''')

# 2. ChatOpenAI는 프롬프트를 요약 메시지로 바꾸고 Parser는 이를 문자열로 변환한다.
summary_llm = ChatOpenAI(
    # model은 `.env`에서 준비한 요약 모델 이름이다.
    model=OPENAI_LLM_MODEL,
    # use_responses_api=True는 Responses API 경로로 모델을 호출한다.
    use_responses_api=True,
    # 낮은 temperature는 같은 원문에서 요약 표현이 크게 달라지는 현상을 줄인다.
    temperature=0.3,
    # reasoning_effort='none'은 요약에 별도의 긴 추론 과정을 사용하지 않는다.
    reasoning_effort="none",
)
summary_chain = summary_prompt | summary_llm | StrOutputParser()

# 3. 첫 문서 원문을 체인에 전달하면 요약 문자열이 반환된다.
text = documents_df.loc[0, "content"]
summary = summary_chain.invoke({"text": text})

print("original characters:", len(text))
print(text)
print("summary characters:", len(summary))
print(summary)


original characters: 191
제주도는 대한민국의 대표 관광지로서, 한라산 등반, 성산 일출봉 관광, 해변 활동(협재해수욕장·함덕해수욕장) 등이 인기입니다. 현지 음식으로는 흑돼지, 고기국수, 전복죽 등이 있으며, 카페 거리(서귀포시 대정읍 카페 거리)도 유명합니다. 교통은 렌터카나 시외버스를 주로 이용하며, 사전 예약 시 우도 투어나 올레길 트레킹도 즐길 수 있습니다.
summary characters: 188
제주도는 대한민국 대표 관광지로, **한라산 등반**, **성산 일출봉**, **협재해수욕장·함덕해수욕장** 해변 활동이 인기다. 대표 음식은 **흑돼지·고기국수·전복죽**이며, **서귀포시 대정읍 카페 거리**도 유명하다. 교통은 **렌터카·시외버스**를 주로 이용하고, 사전 예약하면 **우도 투어·올레길 트레킹**을 즐길 수 있다.


## 전체 문서의 압축본 생성

30개 원문을 같은 요약 체인에 전달해 압축 문서를 만든다. 각 결과는 원래 `doc_id`와 요약 `content`를 함께 저장한다.

`doc_id`는 다음 단계에서 원문과 요약문을 연결하고, Pinecone 검색 결과를 정답 문서 ID와 비교할 때 계속 사용한다.

### 코드 해석 순서

1. content 원문을 같은 summary_chain에 전달해 요약 문자열을 만든다.
2. 원래 doc_id를 보존해 요약문을 원문·정답 데이터와 연결한다.

### 결과 해석

- OpenAI 호출 진행률이 전체 문서 수까지 증가하고 문서별 요약 딕셔너리 목록이 만들어진다.
- 각 요약에 원래 doc_id가 남아 있어 원문·압축 검색을 같은 정답으로 평가할 수 있다.


In [6]:
from tqdm import tqdm

compressed_texts = []
for _, row in tqdm(documents_df.iterrows(), total=len(documents_df)):
    # 1. content 원문을 같은 summary_chain에 전달해 요약 문자열을 만든다.
    compressed_summary = summary_chain.invoke({"text": row["content"]})

    # 2. 원래 doc_id를 보존해 요약문을 원문·정답 데이터와 연결한다.
    compressed_texts.append({
        "doc_id": row["doc_id"],
        "content": compressed_summary,
    })


100%|██████████| 30/30 [01:07<00:00,  2.27s/it]


## 압축본과 원문 나란히 비교

요약 목록을 DataFrame으로 바꾸고 `doc_id`를 기준으로 원문과 병합한다. 병합 결과에서 `content_x`는 압축본, `content_y`는 원문이다.

같은 행의 두 본문을 읽으며 고유명사·수치·조건이 보존됐는지 확인한다.

### 코드 해석 순서

1. 요약 딕셔너리 목록을 `doc_id`, `content` 열의 DataFrame으로 바꾼다.
2. 같은 doc_id의 요약과 원문을 연결하면 content_x와 content_y 열이 만들어진다.

### 결과 해석

- doc_id, 압축 본문 content_x와 원문 content_y를 가진 앞 5개 행이 표시된다.
- 같은 문서의 압축 전후 내용을 직접 비교해 검색 핵심 정보가 유지됐는지 확인할 수 있다.


In [7]:
# 1. 요약 딕셔너리 목록을 `doc_id`, `content` 열의 DataFrame으로 바꾼다.
compressed_df = pd.DataFrame(compressed_texts)

# 2. 같은 doc_id의 요약과 원문을 연결하면 content_x와 content_y 열이 만들어진다.
merged_df = pd.merge(
    compressed_df,
    documents_df[["doc_id", "content"]],
    on="doc_id",
)

# 긴 원문과 요약을 생략하지 않고 표시한다.
pd.set_option("display.max_colwidth", None)
merged_df.head()


,doc_id,content_x,content_y
0,D1,"제주도는 대한민국 대표 관광지로, **한라산 등반**, **성산 일출봉**, **협재해수욕장·함덕해수욕장** 해변 활동이 인기다. 대표 음식은 **흑돼지·고기국수·전복죽**이며, **서귀포시 대정읍 카페 거리**도 유명하다. 교통은 **렌터카·시외버스**를 주로 이용하고, **사전 예약**을 통해 **우도 투어·올레길 트레킹**을 즐길 수 있다.","제주도는 대한민국의 대표 관광지로서, 한라산 등반, 성산 일출봉 관광, 해변 활동(협재해수욕장·함덕해수욕장) 등이 인기입니다. 현지 음식으로는 흑돼지, 고기국수, 전복죽 등이 있으며, 카페 거리(서귀포시 대정읍 카페 거리)도 유명합니다. 교통은 렌터카나 시외버스를 주로 이용하며, 사전 예약 시 우도 투어나 올레길 트레킹도 즐길 수 있습니다."
1,D2,"비빔밥은 조선 시대부터 전해진 대표적 한국 음식으로, 밥에 채소·고기·계란 등 고명을 올린 뒤 고추장이나 간장을 섞어 먹는다. 전주 비빔밥은 다양한 고명과 전주식 고추장을 사용하며 잔치 음식으로도 유명하다. 진주 비빔밥은 고기·회·나물 등을 섞어 풍부한 식감을 내며, 두 지역은 역사적 배경과 재료 구성에 따라 맛과 풍미가 다르다.","비빔밥은 조선 시대부터 전해 내려온 대표적 한국 음식으로, 밥 위에 고명(채소·고기·계란 등)을 올리고 고추장이나 간장을 섞어 먹습니다. 전주 비빔밥은 고명 종류가 다양하고 전주식 고추장을 쓰며, 잔치용으로도 유명합니다. 진주 비빔밥은 고기·회·나물 등을 섞어 더욱 풍부한 식감을 제공합니다. 두 지역 모두 역사적 배경과 재료 구성이 달라 맛과 풍미가 다릅니다."
2,D3,"걸스데이는 2010년 데뷔한 대한민국 4인조 걸그룹으로, 대표곡은 “Something”, “Darling”, “Expectation”이다. 청순 컨셉에서 섹시·여성미 컨셉으로 변화해 음원 차트 상위권에 올랐으며, 민아·유라·소진·혜리는 드라마·예능·광고 등으로 활동 영역을 확장했다.","걸스데이는 2010년 데뷔한 대한민국의 4인조 걸그룹으로, 대표곡으로는 “Something”, “Darling”, “Expectation” 등이 있습니다. 데뷔 초기 청순 컨셉에서 점차 섹시·여성미 컨셉으로 변화하며 음원 차트 상위권에 올랐습니다. 멤버 민아·유라·소진·혜리는 드라마·예능·광고 등 다양한 분야에도 진출해 활동 영역을 넓혔습니다."
3,D4,"세종대왕(1397~1450)은 조선의 4대 임금으로, 백성의 문맹 문제 해결과 국가 통치 효율화를 위해 훈민정음을 창제·보급했다. 그의 업적은 한국 문화와 문자 체계에 큰 영향을 미쳤으며, 훈민정음 해례본은 유네스코 세계기록유산으로 등재되었다.","세종대왕(1397~1450)은 훈민정음을 창제하여 한글을 보급한 조선의 4대 임금입니다. 그가 훈민정음을 만든 배경에는 백성들의 문맹 문제 해결과 국가 통치 효율화가 있었습니다. 세종대왕의 업적은 한국 문화와 문자 체계에 지대한 영향을 미쳤으며, 훈민정음 해례본은 유네스코 세계기록유산으로 등재되었습니다."
4,D5,"이순신 장군(1545~1598)은 임진왜란 당시 명량 해전에서 13척으로 133척의 왜선을 격파했다. 학익진과 기상·해류를 활용한 전략적 승리로, 한국 해군 전통과 군사 전략 연구의 핵심 사례다.",이순신 장군(1545~1598)은 임진왜란 당시 명량 해전에서 13척의 배로 133척의 왜선을 격파하면서 크게 승리했습니다. 전술적인 배 배치(학익진)와 기상·해류를 활용한 전략은 전투 역사에 길이 남을 전술입니다. 이순신의 업적은 한국 해군 전통과 군사 전략 연구에서 핵심 사례로 다뤄집니다.


## 압축 문서 전용 Pinecone 인덱스

원문과 요약문을 섞지 않도록 `adv-rag-compressed` index를 별도로 준비한다.

- `name`: Pinecone에서 index를 식별하는 이름이다.
- `dimension`: 저장할 임베딩 벡터의 길이이다.
- `metric`: 질문 벡터와 문서 벡터의 유사도를 계산하는 방법이다.
- `ServerlessSpec`: index를 배치할 cloud와 region을 지정한다.

index가 없다면 생성하고, 생성이 끝난 뒤 검색 가능한 `Ready` 상태가 될 때까지 기다린다.

### 코드 해석 순서

1. 원문 index와 같은 차원·metric을 사용해 압축 문서 전용 index를 생성한다.
2. 생성 직후에는 준비 시간이 필요하므로 Ready 상태가 될 때까지 1초 간격으로 확인한다.

### 결과 해석

- Pinecone에서 압축 index를 새로 생성하거나 기존 index를 재사용했다는 메시지가 출력된다.
- Ready 상태가 된 index에만 다음 셀의 요약 Document를 안전하게 저장할 수 있다.


In [8]:
import time

from pinecone import Pinecone, ServerlessSpec

pc = Pinecone()
existing_indexes = pc.list_indexes().names()

if PINECONE_COMP_INDEX_NAME not in existing_indexes:
    # 1. 원문 index와 같은 차원·metric을 사용해 압축 문서 전용 index를 생성한다.
    pc.create_index(
        name=PINECONE_COMP_INDEX_NAME,
        dimension=PINECONE_INDEX_DIMENSION,
        metric=PINECONE_INDEX_METRIC,
        # spec은 serverless index를 배치할 cloud와 region 설정을 받는다.
        spec=ServerlessSpec(
            cloud=PINECONE_INDEX_CLOUD,
            region=PINECONE_INDEX_REGION,
        ),
    )

    # 2. 생성 직후에는 준비 시간이 필요하므로 Ready 상태가 될 때까지 1초 간격으로 확인한다.
    while not pc.describe_index(PINECONE_COMP_INDEX_NAME).status["ready"]:
        time.sleep(1)
    print(f"{PINECONE_COMP_INDEX_NAME} index를 생성했다.")
else:
    print(f"{PINECONE_COMP_INDEX_NAME} index를 재사용한다.")


adv-rag-compressed index를 생성했다.


## 원문·압축 벡터 스토어 연결

하나의 `OpenAIEmbeddings` 객체를 원문 index와 압축 index에 공통으로 사용한다. 두 `PineconeVectorStore`는 `index_name`만 다르다.

동일한 임베딩 모델과 질문으로 검색하면 두 결과의 차이는 index에 저장한 원문과 요약문 표현에서 발생한다.

### 코드 해석 순서

1. 같은 임베딩 모델로 원문·압축 Pinecone Vector Store를 연결한다.

### 결과 해석

- 두 PineconeVectorStore 객체가 생성되며 이 셀에는 별도 검색 출력이 나타나지 않는다.
- 같은 질문 벡터로 원문과 요약문 index를 검색할 준비가 완료된다.


In [9]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

# dimensions는 Pinecone index를 만들 때 지정한 벡터 길이와 같아야 한다.
embeddings = OpenAIEmbeddings(
    model=OPENAI_EMBEDDING_MODEL,
    dimensions=PINECONE_INDEX_DIMENSION,
)

# 원문 스토어는 기존 index를, 압축 스토어는 앞에서 준비한 별도 index를 사용한다.
vector_store = PineconeVectorStore(
    index_name=PINECONE_INDEX_NAME,
    embedding=embeddings,
)
comp_vector_store = PineconeVectorStore(
    index_name=PINECONE_COMP_INDEX_NAME,
    embedding=embeddings,
)


## 압축 Document 일괄 upsert

각 요약 행을 `Document`로 바꾸고 압축 index에 저장한다.

- `page_content`: 임베딩할 요약 본문이다.
- `metadata['doc_id']`: 검색 결과를 원래 문서와 정답 데이터로 연결한다.
- `ids`: Pinecone 레코드 ID이다. 원래 `doc_id`를 사용하면 셀을 다시 실행해도 같은 레코드를 갱신한다.

### 코드 해석 순서

1. 각 요약 행을 PineconeVectorStore가 받을 Document로 변환한다.
2. documents와 ids의 같은 위치가 하나의 Pinecone 레코드로 저장된다.

### 결과 해석

- 임베딩과 upsert가 끝나면 저장한 문서 개수가 출력된다.
- 같은 고정 ID로 다시 실행하면 중복 레코드를 추가하지 않고 기존 압축 문서를 갱신한다.


In [10]:
from langchain_core.documents import Document

# 1. 각 요약 행을 PineconeVectorStore가 받을 Document로 변환한다.
comp_docs = []
comp_ids = []
for _, row in compressed_df.iterrows():
    doc_id = str(row["doc_id"])
    comp_docs.append(
        Document(
            page_content=row["content"],
            metadata={"doc_id": doc_id},
        )
    )
    comp_ids.append(doc_id)

# 2. documents와 ids의 같은 위치가 하나의 Pinecone 레코드로 저장된다.
upserted_ids = comp_vector_store.add_documents(
    documents=comp_docs,
    ids=comp_ids,
)

print("upserted document count:", len(upserted_ids))


upserted document count: 30


## 원문·압축 검색 평가 함수

두 결과에 동일한 P@k, R@k, RR와 AP@k를 적용한다. 함수 입력은 순서가 있는 문서 ID 목록과 질의별 관련 문서 딕셔너리이고, 반환값은 전체 질의의 평균 지표이다. AP@k 분모에 놓친 관련 문서가 남도록 계산해 요약 손실을 과소평가하지 않는다.

### 코드 해석 순서

1. qrels 문자열을 문서 ID와 관련성 등급의 딕셔너리로 변환한다.
2. 예측 문서 ID 순서와 qrels 딕셔너리에서 질의 하나의 네 지표를 계산한다.
3. 모든 질의의 지표를 평균내 두 index의 비교표가 사용할 딕셔너리를 반환한다.

### 결과 해석

- 평가 함수 정의만 수행되며 마지막 표 셀이 이를 호출한다.
- 문자 수 감소가 검색 품질 하락으로 이어지는지는 Recall@5와 MAP를 함께 봐야 한다.


In [11]:
import numpy as np


# 1. qrels 문자열을 문서 ID와 관련성 등급의 딕셔너리로 변환한다.
def parse_relevant(relevant_str):
    # 입력 예시 `D6=3;D14=2`를 문서 ID와 관련성 등급의 딕셔너리로 변환한다.
    relevant_dict = {}
    for pair in relevant_str.split(";"):
        doc_id, grade_text = pair.split("=")
        grade = int(grade_text)
        # 0등급 문서가 포함되더라도 정답 hit로 세지 않고 1 이상만 보존한다.
        if grade > 0:
            relevant_dict[doc_id] = grade
    return relevant_dict


# 2. 예측 문서 ID 순서와 qrels 딕셔너리에서 질의 하나의 네 지표를 계산한다.
def compute_metrics(predicted, relevant_dict, k=5):
    # predicted[:k]가 평가 대상이며 관련성 등급이 1 이상인 문서를 정답으로 처리한다.
    top_k = predicted[:k]
    hits = sum(doc_id in relevant_dict for doc_id in top_k)
    precision = hits / k

    total_relevant = len(relevant_dict)
    recall = hits / total_relevant if total_relevant else 0.0

    # RR은 첫 관련 문서의 순위 역수이므로 첫 정답이 1위이면 1.0이다.
    rr = next(
        (1 / rank for rank, doc_id in enumerate(predicted, start=1) if doc_id in relevant_dict),
        0.0,
    )

    # AP@k는 관련 문서를 만난 각 순위의 Precision을 min(관련 문서 수, k)로 나눈다.
    precision_sum = 0.0
    relevant_seen = 0
    for rank, doc_id in enumerate(top_k, start=1):
        if doc_id in relevant_dict:
            relevant_seen += 1
            precision_sum += relevant_seen / rank
    ap_denominator = min(total_relevant, k)
    ap = precision_sum / ap_denominator if ap_denominator else 0.0
    return precision, recall, rr, ap


# 3. 모든 질의의 지표를 평균내 두 index의 비교표가 사용할 딕셔너리를 반환한다.
def evaluate_all(method_results, queries_df, k=5):
    # 각 질의의 네 지표를 누적한 뒤 평균을 반환해 검색기 간 비교표에 사용한다.
    per_query_metrics = []
    for _, row in queries_df.iterrows():
        relevant_dict = parse_relevant(row["relevant_doc_ids"])
        predicted = method_results[row["query_id"]]
        per_query_metrics.append(compute_metrics(predicted, relevant_dict, k))

    metric_array = np.asarray(per_query_metrics, dtype=float)
    return {
        "P@k": metric_array[:, 0].mean(),
        "R@k": metric_array[:, 1].mean(),
        "MRR": metric_array[:, 2].mean(),
        "MAP": metric_array[:, 3].mean(),
    }


## 동일 질의로 두 인덱스 검색

30개 질문을 원문과 압축 벡터 스토어에 각각 보내 상위 5개 문서 ID를 수집한다.

- 동일한 요소: 질문, 임베딩 모델, `k=5`, 정답 데이터이다.
- 다른 요소: Pinecone에 저장된 원문과 요약문 표현이다.

따라서 두 결과의 순위 차이를 문서 압축이 임베딩과 검색에 미친 영향으로 해석할 수 있다.

### 코드 해석 순서

1. 원문 index의 list[Document]를 문서 ID 순서로 변환한다.
2. 같은 질문으로 압축 index를 검색하고 동일한 문서 ID 형식으로 맞춘다.

### 결과 해석

- Pinecone 검색이 끝나면 질의 ID별 원문·압축 상위 문서 ID 딕셔너리가 만들어진다.
- 출력 형식을 같게 만들면 저장 본문만 다른 두 검색 결과를 동일 지표로 평가할 수 있다.


In [12]:
dense_results = {}
comp_results = {}

for _, row in tqdm(queries_df.iterrows(), total=len(queries_df)):
    qid = row["query_id"]
    query_text = row["query_text"]

    # 1. 원문 index의 list[Document]를 문서 ID 순서로 변환한다.
    dense_docs = vector_store.similarity_search(query_text, k=5)
    dense_results[qid] = [doc.metadata["doc_id"] for doc in dense_docs]

    # 2. 같은 질문으로 압축 index를 검색하고 동일한 문서 ID 형식으로 맞춘다.
    compressed_docs = comp_vector_store.similarity_search(query_text, k=5)
    comp_results[qid] = [doc.metadata["doc_id"] for doc in compressed_docs]


100%|██████████| 30/30 [00:38<00:00,  1.28s/it]


## 원문과 압축 인덱스 지표 비교

두 결과의 P@5, R@5, MRR와 MAP를 한 표로 비교한다. 요약으로 문서가 짧아져도 검색 핵심 정보가 사라지면 Recall이나 순위 지표가 낮아질 수 있다.

HyDE·Rerank·문서 압축은 순차 단계가 아니라 서로 다른 검색 병목에 맞춰 고르는 선택지이다. 질문 표현이 약하면 HyDE, 후보 내부 순서가 약하면 Rerank, 인덱싱할 문서 표현이 길거나 중복이 많으면 문서 압축을 검토한다.

### 코드 해석 순서

1. 같은 queries_df와 k=5 기준으로 두 검색 결과의 평균 지표를 계산한다.
2. 같은 지표를 한 행에 배치해 요약 전후의 검색 품질 차이를 확인한다.

### 결과 해석

- 외부 호출을 모두 실행하면 원문과 압축 검색의 P@5·R@5·MRR·MAP 표가 표시된다.
- 압축의 효과는 문자 수 감소만으로 판단하지 않고 검색 지표와 정보 손실을 함께 비교해야 한다.


In [13]:
# 1. 같은 queries_df와 k=5 기준으로 두 검색 결과의 평균 지표를 계산한다.
dense_metrics = evaluate_all(dense_results, queries_df)
comp_metrics = evaluate_all(comp_results, queries_df)

# 2. 같은 지표를 한 행에 배치해 요약 전후의 검색 품질 차이를 확인한다.
metrics_df = pd.DataFrame({
    "Metric": ["P@5", "R@5", "MRR", "MAP"],
    "Dense": [
        dense_metrics["P@k"],
        dense_metrics["R@k"],
        dense_metrics["MRR"],
        dense_metrics["MAP"],
    ],
    "Compressed": [
        comp_metrics["P@k"],
        comp_metrics["R@k"],
        comp_metrics["MRR"],
        comp_metrics["MAP"],
    ],
})
metrics_df


,Metric,Dense,Compressed
0,P@5,0.260000,0.266667
1,R@5,0.916667,0.927778
2,MRR,0.983333,0.983333
3,MAP,0.880741,0.881852
